<a href="https://colab.research.google.com/github/Hartheek1704/house-price-prediction-ml/blob/main/Task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
df = pd.read_csv("/content/Housing.csv")

print("Dataset Preview:")
print(df.head())


Dataset Preview:
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  


In [3]:
df['UserID'] = np.random.randint(1, 21, size=len(df))  # 20 users

df['ItemID'] = df.index

df['Rating'] = df['price'] / df['price'].max()

data = df[['UserID', 'ItemID', 'Rating']]

In [4]:
user_item_matrix = data.pivot_table(index='UserID', columns='ItemID', values='Rating').fillna(0)

print("\nUser-Item Matrix:")
print(user_item_matrix.head())


User-Item Matrix:
ItemID   0    1         2    3         4    5         6    7    8         9   \
UserID                                                                         
1       0.0  0.0  0.921053  0.0  0.000000  0.0  0.000000  0.0  0.0  0.736842   
3       0.0  0.0  0.000000  0.0  0.857895  0.0  0.000000  0.0  0.0  0.000000   
4       0.0  0.0  0.000000  0.0  0.000000  0.0  0.000000  0.0  0.0  0.000000   
5       0.0  0.0  0.000000  0.0  0.000000  0.0  0.000000  0.0  0.0  0.000000   
6       0.0  0.0  0.000000  0.0  0.000000  0.0  0.763158  0.0  0.0  0.000000   

ItemID  ...   39   40   41   42        43   44   45   46   47   48  
UserID  ...                                                         
1       ...  0.0  0.0  0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  
3       ...  0.0  0.0  0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  
4       ...  0.0  0.0  0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  
5       ...  0.0  0.0  0.0  0.0  0.578947  0.0  0.0  0.0  0.0  0.0  
6     

In [5]:
item_similarity = cosine_similarity(user_item_matrix.T)

item_similarity_df = pd.DataFrame(item_similarity,
                                 index=user_item_matrix.columns,
                                 columns=user_item_matrix.columns)

In [6]:
def recommend_items(user_id, top_n=5):
    user_ratings = user_item_matrix.loc[user_id]


    rated_items = user_ratings[user_ratings > 0].index

    scores = {}

    for item in rated_items:
        similar_items = item_similarity_df[item]

        for sim_item, score in similar_items.items():
            if sim_item not in rated_items:
                scores[sim_item] = scores.get(sim_item, 0) + score


    recommended_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return recommended_items[:top_n]


In [7]:
user_id = 5
recommendations = recommend_items(user_id)

print(f"\nTop Recommendations for User {user_id}:")
for item, score in recommendations:
    print(f"House ID: {item}, Score: {score:.4f}")


Top Recommendations for User 5:
House ID: 0, Score: 0.0000
House ID: 1, Score: 0.0000
House ID: 2, Score: 0.0000
House ID: 3, Score: 0.0000
House ID: 4, Score: 0.0000


In [8]:
from sklearn.metrics import mean_squared_error

# Dummy prediction vs actual (example)
y_true = data['Rating']
y_pred = np.random.rand(len(y_true))

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print("RMSE:", rmse)

RMSE: 0.36597995688968227
